# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Wun-nam Haruna]
**Student ID:** [10032027]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [1]:
# API-key setup — DO NOT hard-code your key in this cell.

import os
from getpass import getpass
from openai import OpenAI

# Safer local method:
# This asks for your API key at runtime without saving it inside the notebook.
API_KEY = os.environ.get("GROQ_API_KEY")

if API_KEY is None:
    API_KEY = getpass("Enter your GROQ_API_KEY: ")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

MODEL = "llama-3.1-8b-instant"

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [2]:
def ask_llm(

    user_prompt,

    system_prompt="You are a helpful assistant.",

    temperature=0.7,

    max_tokens=500,

):

    response = client.chat.completions.create(

        model=MODEL,

        messages=[

            {"role": "system", "content": system_prompt},

            {"role": "user", "content": user_prompt},

        ],

        temperature=temperature,

        max_tokens=max_tokens,

    )

    return response.choices[0].message.content

# First API call

answer = ask_llm(

    "In one short paragraph, explain what an API call to an LLM does."

)

print(answer)

An API call to a Large Language Model (LLM) is a request sent to the model's server to process a specific input, such as text or a prompt. This call initiates a computation that generates a response, which is then returned to the user. The response can be a generated text, a classification, a prediction, or an answer to a question, depending on the model's capabilities and the input provided. The API call typically includes parameters specifying the input data, model configuration, and desired output format.


**Student Reasoning — Anatomy of a call**

**Question 1: What is the difference between the `system` and `user` roles? Give an example of something that belongs in each.**

The `system` role gives the model its overall instructions. It tells the model how it should behave before answering the user.

Example system message:

`You are a helpful assistant who explains things simply.`

The `user` role is the actual request or question from the person using the model.

Example user message:

`Explain what an API call to an LLM does in one short paragraph.`

So the system message controls the style or rules, while the user message gives the specific task.

**Question 2: What is a token, roughly? Why do API providers bill per token rather than per request?**

A token is a small piece of text. It can be a word, part of a word, a number, or punctuation.

API providers bill per token because longer prompts and longer answers require more computation. A short request uses fewer tokens and costs less, while a long request with a long answer uses more tokens and costs more.

So billing per token is fairer than billing only per request because not all requests require the same amount of work.

### Part 1.2 — Temperature: the randomness dial

In [3]:
test_question = "Suggest a name for a savings product for market traders in Accra."

print("Temperature = 0.0")
print("-" * 40)

for i in range(5):
    answer = ask_llm(
        test_question,
        system_prompt="You are a creative but clear product naming assistant.",
        temperature=0.0,
        max_tokens=80,
    )
    print(f"{i + 1}. {answer}\n")


print("\nTemperature = 1.2")
print("-" * 40)

for i in range(5):
    answer = ask_llm(
        test_question,
        system_prompt="You are a creative but clear product naming assistant.",
        temperature=1.2,
        max_tokens=80,
    )
    print(f"{i + 1}. {answer}\n")

Temperature = 0.0
----------------------------------------
1. Considering the target audience and the product's purpose, here are some name suggestions for a savings product for market traders in Accra:

1. **MarketMate Savings**: This name emphasizes the product's connection to the market traders and their daily lives.
2. **Accra Prosper**: This name conveys the idea of growth and prosperity, which is essential for market traders looking to save and invest.
3

2. Considering the target audience and the product's purpose, here are some name suggestions for a savings product for market traders in Accra:

1. **MarketMate Savings**: This name emphasizes the product's connection to the market traders and their daily lives.
2. **Accra Prosper**: This name conveys the idea of growth and prosperity, which is essential for market traders looking to save and invest.
3

3. Considering the target audience and the product's purpose, here are some name suggestions for a savings product for market t

**Student Reasoning — Temperature**

**Question: What did you observe at each temperature? For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?**

At `temperature = 0.0`, the answers were very similar each time. The model was more predictable and gave almost the same type of product name repeatedly.

At `temperature = 1.2`, the answers were more varied and creative. The model gave different names and sometimes used more imaginative wording.

For a loan decision-support system, a low temperature is more appropriate. The system should be consistent, careful, and reliable because loan decisions affect people financially. We do not want the model to give very different advice each time for the same applicant. A low temperature helps make the output more stable and easier to trust.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [6]:
# Prompt V1: naive prompt
SUMMARY_PROMPT_V1 = "Summarize this:"
import pandas as pd

def summarize_v1(letter_text):
    return ask_llm(
        user_prompt=f"{SUMMARY_PROMPT_V1}\n\n{letter_text}",
        system_prompt="You are a helpful assistant.",
        temperature=0.7,
        max_tokens=250,
    )


# Prompt V2: better structured prompt
SUMMARY_SYSTEM_PROMPT_V2 = """
You are an assistant to a microfinance loan officer in Ghana.
Your task is to summarize loan application letters clearly and neutrally.

Rules:
- Use only facts stated in the letter.
- Do not invent missing details.
- Mention the applicant, amount requested, purpose, income/profit if stated, repayment plan, and collateral/guarantor if stated.
- Keep the summary to 3-4 sentences.
- Use a professional and factual tone.
"""

def summarize_v2(letter_text):
    return ask_llm(
        user_prompt=f"Summarize this loan application:\n\n{letter_text}",
        system_prompt=SUMMARY_SYSTEM_PROMPT_V2,
        temperature=0.0,
        max_tokens=250,
    )


# Run both prompt versions on L002 and L006
letters_to_test = ["L002", "L006"]

summary_results = []

for letter_id in letters_to_test:
    letter_text = LETTERS[letter_id]

    v1_output = summarize_v1(letter_text)
    v2_output = summarize_v2(letter_text)

    summary_results.append(
        {
            "letter_id": letter_id,
            "V1_naive_output": v1_output,
            "V2_structured_output": v2_output,
        }
    )

summary_df = pd.DataFrame(summary_results)

pd.set_option("display.max_colwidth", None)
summary_df

,letter_id,V1_naive_output,V2_structured_output
0,L002,"Kwame Boateng, a commercial driver in Kumasi, Ghana, is seeking GHS 25,000 urgently to repair his trotro engine and settle personal debts. He expects business to improve after the festive season, assuring that he will repay the loan when funds become available. Unfortunately, he has no collateral to secure the loan.","Loan Application Summary:\n\nKwame Boateng, a commercial driver in Kumasi, is requesting a loan of GHS 25,000. The funds will be used to repair his trotro engine and settle personal debts. Mr. Boateng's business has been slow, but he anticipates an increase in revenue after the festive season. He does not have collateral to offer at this time."
1,L006,"Kofi has applied for a loan of GHS 50,000, stating he wants to start three separate businesses: a car washing service, a provision shop, and importing phones from Dubai. He claims to be 22 years old, full of energy, and business-minded, but has no prior experience with any of these ventures. He offers to repay the loan in one year, without providing collateral, and asserts that he is trustworthy.","Loan Application Summary:\n\nKofi, a 22-year-old, has applied for a loan of GHS 50,000 to start a car washing business, a provision shop, and import phones from Dubai. The loan is intended to be repaid within one year, once the businesses are established. Kofi does not offer any collateral but claims to be trustworthy. The repayment plan is based on the assumption that the businesses will be profitable within a year."


**Student Reasoning — Summarization prompts**

**Question 1: What concrete problems did V1's output have that V2 fixed? Quote examples.**

V1 gave a general summary, but it was less controlled and less useful for a loan officer. For example, for `L002`, V1 says the applicant is asking for money to repair his trotro engine and settle personal debts, but it does not clearly organize the key loan details.

V2 is better because it follows the loan officer’s needs more directly. For `L002`, V2 clearly states the loan amount, purpose, business situation, repayment plan, and lack of collateral. It says the applicant requests `GHS 25,000`, needs it to repair his trotro engine and settle debts, expects to repay when money becomes available, and has no collateral.

For `L006`, V1 says Kofi wants to start three businesses and has no prior experience. V2 is more careful and structured because it states the amount requested, the three proposed businesses, the repayment period, and the risk that he has no collateral or proven business operation.

**Question 2: Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?**

"No invented details" is essential because this is a loan decision-support system. If the model adds information that the applicant did not state, it could unfairly affect whether the person gets a loan.

For example, if the model invents collateral, income, or repayment ability, the loan officer may think the applicant is safer than they really are. If it invents negative details, it could unfairly hurt the applicant.

This failure mode is called **hallucination** in LLM literature. It means the model produces information that sounds confident but is not actually supported by the source text.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [7]:
import json
import pandas as pd

EXTRACT_SYSTEM_PROMPT = """
You are an information extraction assistant for a microfinance loan officer.

Return ONLY a valid JSON object.
Do not include markdown.
Do not include explanations.
Do not include ```json fences.

The JSON object must have EXACTLY these keys:
{
  "applicant_name": string,
  "amount_ghs": number,
  "purpose": string,
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": boolean,
  "repayment_months": number or null
}

Rules:
- Use only information stated in the letter.
- If a field is not stated in the letter, use null.
- Do not guess missing information.
- For has_collateral_or_guarantor, use true only if the letter clearly mentions collateral, a guarantor, group backing, joint liability, or a similar repayment support.
- Return numbers without currency symbols or commas.
"""

EXTRACT_EXAMPLE = """
Example letter:
Dear Loan Officer,
My name is Ama Tetteh. I run a small food stall in Madina and I request GHS 6,000 to buy a new stove and more cooking ingredients. My monthly profit is about GHS 1,200. My brother has agreed to guarantee the loan. I can repay over 10 months.

Correct JSON:
{
  "applicant_name": "Ama Tetteh",
  "amount_ghs": 6000,
  "purpose": "buy a new stove and more cooking ingredients",
  "monthly_profit_ghs": 1200,
  "has_collateral_or_guarantor": true,
  "repayment_months": 10
}
"""


def clean_json_text(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "", 1).strip()

    if text.startswith("```"):
        text = text.replace("```", "", 1).strip()

    if text.endswith("```"):
        text = text[:-3].strip()

    return text


def extract_fields(letter_text):
    user_prompt = f"""
{EXTRACT_EXAMPLE}

Now extract the same fields from this loan application letter:

{letter_text}
"""

    raw_output = ask_llm(
        user_prompt=user_prompt,
        system_prompt=EXTRACT_SYSTEM_PROMPT,
        temperature=0.0,
        max_tokens=300,
    )

    cleaned_output = clean_json_text(raw_output)

    try:
        data = json.loads(cleaned_output)
        return data

    except json.JSONDecodeError:
        print("Warning: Could not parse JSON output.")
        print("Raw output:")
        print(raw_output)
        return None


# Run extraction on all six letters
extraction_results = []

for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)

    if extracted is None:
        extracted = {
            "applicant_name": None,
            "amount_ghs": None,
            "purpose": None,
            "monthly_profit_ghs": None,
            "has_collateral_or_guarantor": None,
            "repayment_months": None,
        }

    extracted["letter_id"] = letter_id
    extraction_results.append(extracted)


extraction_df = pd.DataFrame(extraction_results)

# Put letter_id first
cols = ["letter_id"] + [col for col in extraction_df.columns if col != "letter_id"]
extraction_df = extraction_df[cols]

extraction_df

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some personal debts,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fabric stock,2800.0,True,15.0
3,L004,Yaw Owusu,12000,feed and 500 new layers,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the factory,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop, and also import phones from Dubai",NaN,False,12.0


**Student Reasoning — Structured extraction**

**Question 1: Why must the few-shot example NOT come from the six letters you are processing?**

The few-shot example should not come from the six letters because that would leak the answer into the prompt. If we use one of the real letters as the example, the model is no longer being tested fairly on unseen letters.

The example should teach the model the format, not give it one of the actual cases it is supposed to extract from.

**Question 2: Why "use null, do not guess" — what did the model do without that instruction?**

We use `null` because some letters do not state every field clearly. For example, some applicants do not state monthly profit or exact repayment details.

Without that instruction, the model may guess missing values or assume information that is not written in the letter. That is dangerous because the extraction table could look complete even when the original letter did not provide the information.

**Question 3: Why is temperature=0 the right choice for extraction but arguably not for creative tasks?**

`temperature=0` is best for extraction because we want consistent and factual answers. The model should return the same structured fields every time and avoid creative wording.

For creative tasks, a higher temperature can be useful because it gives more variety and more original ideas. But for extraction, creativity is a problem because the model may change formats, invent details, or become inconsistent.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [8]:
BRIEF_SYSTEM_PROMPT = """
You are an assistant to a microfinance loan officer in Ghana.

Your task is to prepare a decision-support brief.
You must support the human loan officer, not make the final decision.

Rules:
- Use only information from the loan letter and extracted JSON.
- Do not invent missing facts.
- Be neutral and professional.
- Do not say "approve" or "reject".
- The final decision must be made by a human loan officer.
- If information is missing, clearly list what should be requested.
"""

def make_brief(letter_id, letter_text, extracted_json):
    user_prompt = f"""
Prepare a loan decision-support brief for this application.

Letter ID: {letter_id}

Extracted JSON:
{json.dumps(extracted_json, indent=2)}

Original loan application letter:
{letter_text}

Output format:

1. Strengths
- bullet points

2. Risks / red flags
- bullet points

3. Missing information the officer should request
- bullet points

4. Suggested next step
- one short recommendation such as "invite for interview", "request documents", or "flag for senior review"
"""

    brief = ask_llm(
        user_prompt=user_prompt,
        system_prompt=BRIEF_SYSTEM_PROMPT,
        temperature=0.0,
        max_tokens=500,
    )

    return brief


# Generate briefs for all six letters
brief_results = {}

for letter_id, letter_text in LETTERS.items():
    row = extraction_df[extraction_df["letter_id"] == letter_id]

    if len(row) == 0:
        print(f"Warning: No extracted JSON found for {letter_id}")
        continue

    extracted_json = row.drop(columns=["letter_id"]).iloc[0].to_dict()

    brief_results[letter_id] = make_brief(
        letter_id=letter_id,
        letter_text=letter_text,
        extracted_json=extracted_json,
    )


# Print briefs for three different applications
for letter_id in ["L001", "L002", "L006"]:
    print("=" * 80)
    print(f"Decision-support brief for {letter_id}")
    print("=" * 80)
    print(brief_results[letter_id])
    print()

Decision-support brief for L001
**Loan Decision-Support Brief**

**Letter ID:** L001

**Applicant:** Akosua Mensah

**1. Strengths:**
- The applicant has a stable income source from selling provisions at Makola Market for 12 years.
- The applicant has a history of saving with the susu scheme, demonstrating financial discipline.
- The applicant has a guarantor, which can provide additional security for the loan.
- The applicant has a clear plan for using the loan to expand their business.

**2. Risks / Red Flags:**
- The applicant's monthly profit is relatively low (GHS 900), which may impact their ability to repay the loan.
- The applicant's business expansion plan relies on a single asset (deep freezer), which may not be a diversified risk.
- The applicant's guarantor is a family member, which may not provide the same level of security as an independent guarantor.

**3. Missing Information the Officer Should Request:**
- Detailed financial statements for the applicant's current busine

**Student Reasoning — Decision support**

**Question 1: Compare the briefs for L003 and L006. Did the system identify the right strengths and red flags in each?**

Yes, the system identified the main differences between the stronger and weaker applications.

For `L003`, the application is stronger because Efua Darko has a registered dressmaking business, employs three apprentices, has past revenue records, has a fixed deposit, and proposes a clear repayment plan. These are useful strengths because they show business history, some financial evidence, and repayment planning.

For `L006`, the application is weaker because Kofi is asking for a large loan of `GHS 50,000` to start several businesses at once, but he has not started them yet. He also has no collateral and only says he is trustworthy. The system correctly treats this as risky because the repayment depends on future business success that has not been proven.

**Question 2: Why did we forbid the model from outputting "approve" or "reject"? Give one practical and one ethical reason.**

A practical reason is that the model does not have all the information needed to make a final loan decision. A human loan officer still needs to verify documents, check income, assess risk, and follow the institution’s rules.

An ethical reason is that loan decisions affect people’s lives and businesses. It would be unfair to let an AI system make the final decision without human review, especially because the model can make mistakes or miss context.

So the model should support the decision by summarizing strengths, risks, and missing information, but the final decision should stay with a human loan officer.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

### Part 4.2 — Reliability: is the system consistent?

In [ ]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.